# Linear Regression

回归（regression）是能为一个或多个自变量与因变量之间关系建模的一类方法，回归可以用于预测多少的问题

$$y = w \cdot x + b$$

$w$ (Weight，权重): 相当于直线的斜率，它决定了输入 $x$ 有多重要

$b$ (Bias，偏置): 相当于截距

在深度学习视角中：

1. 输入层: 数据 $x$ 进入网络

2. 计算层: 神经元进行计算 $w \cdot x + b$

3. 输出层: 得到预测值 $\hat{y}$

我们用 Loss 评判 $w$ 和 $b$ 选得好不好

线性回归最常用的是均方误差

$$Loss = \frac{1}{2} \sum (y_{真实} - y_{预测})^2$$

如何让 Loss 变小，使用梯度下降方法

1. 求导: 计算当前位置的坡度

2. 学习率: 决定你下山一步迈多大

3. 更新参数: $$w_{新} = w_{旧} - \text{学习率} \times \text{梯度}$$


梯度下降：

全量：如果你有 1000 万张图片，内存瞬间爆炸，算一次梯度要等很久

随机：每次只看一张图，梯度方向乱飞，且无法利用 GPU 的并行计算能力

小批量随机梯度下降：每次抓一小把数据，算出平均方向，然后走一步

In [ ]:
# 线性回归实现
import random
import torch

# 构建数据集
def synthetic_data(w, b, num_examples):
    """生成 y = Xw + b + 噪声"""
    X = torch.normal(
        mean = 0.0,
        std = 1.0,
        size = (num_examples, len(w))
    )
    y = torch.matmul(X, w) + b
    y += torch.normal(
        mean = 0.0,
        std = 0.01,
        size = (y.shape)
    )
    return X, y.reshape((-1, 1))

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

# 数据迭代器
def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices) # 打乱下标
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(
            indices[i:min(i+batch_size, num_examples)]
        )
        yield features[batch_indices], labels[batch_indices]

batch_size = 10
w = torch.normal(0, 0.01, size=(2,1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# 定义模型
def linreg(X, w, b):
    return torch.matmul(X, w) + b

# 定义损失函数
def squared_loss(y_hat, y):
    return (y_hat - y.reshape(y_hat.shape))**2 / 2

# 优化算法
def sgd(params, lr, batch_size):
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

# 训练
lr = 0.03
num_epoch = 4
net = linreg
loss = squared_loss

for epoch in range(num_epoch):
    for X, y in data_iter(batch_size, features, labels):
        l = loss(net(X, w, b), y)
        l.sum().backward()
        sgd([w, b], lr, batch_size)
    with torch.no_grad():
        train_l = loss(net(features, w, b), labels)
        print(f'epoch{epoch + 1}, loss{float(train_l.mean()):f}')


    


epoch1, loss0.056896
epoch2, loss0.000268
epoch3, loss0.000051
epoch4, loss0.000049


In [5]:
import torch
from torch import nn
from torch.utils import data

# 生成合成数据集
def synthetic_data(w, b, num_examples):
    """生成 y = Xw + b + 噪声"""
    X = torch.normal(0, 1, size=(num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, size=y.shape)
    return X, y.reshape((-1, 1))

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

# 读取数据集
def load_array(data_arrays, batch_size, is_train=True):
    """构造一个 PyTorch 数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

# 定义模型
net = nn.Sequential(nn.Linear(2, 1))

# 初始化模型参数
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

# 定义损失函数
loss = nn.MSELoss()

# 定义优化算法
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

# 训练循环
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)     # 前向传播：计算预测值并得到损失
        trainer.zero_grad()      # 梯度清零：防止梯度累加
        l.backward()             # 反向传播：PyTorch 自动计算梯度
        trainer.step()           # 更新参数：执行一步 SGD
    
    # 打印本轮结束后的总损失
    with torch.no_grad():
        current_l = loss(net(features), labels)
        print(f'epoch {epoch + 1}, loss {current_l:f}')

# 最终结果对比
w = net[0].weight.data
print(f'\nw 的估计误差: {true_w - w.reshape(true_w.shape)}')
b = net[0].bias.data
print(f'b 的估计误差: {true_b - b}')


epoch 1, loss 0.000387
epoch 2, loss 0.000088
epoch 3, loss 0.000090

w 的估计误差: tensor([-0.0008,  0.0005])
b 的估计误差: tensor([0.0002])
